# 🐔 jev-guy · 电子斗蛐蛐决策模型（单 Notebook 版）

**Jev 风格决策模型打 2D 格斗游戏**：给模型一个游戏状态（~300 token）和 8 个固定动作，一次前向直接输出每个动作的概率——不写作文，只做选择。

本 notebook 自包含整个项目：物理环境 → 大脑（随机/规则专家/SemIf 零训练/kev 微调）→ 评估 → 录像与视频 → 数据采集 → 训练。

**使用三步**：① 按 `index.md` 装好依赖；② 复制 `project_config.example.yaml` 为 `project_config.yaml` 填好路径；③ Run All。
模型/依赖未配置的章节会**自动跳过**，不影响整本执行。

## 0. 配置加载与环境自检

读取 `project_config.yaml`（没有则退回 example 模板），解析 `results_root` 并创建产物目录。

In [ ]:
import os
import pathlib

import yaml

NB_DIR = pathlib.Path.cwd()  # jupyter 内核的工作目录（应为 notebook 所在文件夹）
_cfg = NB_DIR / "project_config.yaml"
if not _cfg.exists():
    _cfg = NB_DIR / "project_config.example.yaml"
    print("[config] 未找到 project_config.yaml，使用 example 模板（模型章节将自动跳过）")
CFG = yaml.safe_load(open(_cfg, encoding="utf-8"))


def _resolve(p):
    if not p:
        return None
    p = pathlib.Path(os.path.expanduser(str(p)))
    return p if p.is_absolute() else (NB_DIR / p).resolve()


RESULTS = _resolve(CFG["paths"]["results_root"])
for sub in ("replays", "models", "dataset", ".tmp"):
    (RESULTS / sub).mkdir(parents=True, exist_ok=True)
os.environ.setdefault("TMPDIR", str(RESULTS / ".tmp"))  # 某些精简/沙箱环境没有可用系统临时目录
PATHS = {k: _resolve(v) for k, v in CFG["paths"].items() if k != "results_root"}
RUNTIME, DEMO, ARENA = CFG["runtime"], CFG["demo"], CFG["arena"]
print(f"[config] 结果目录: {RESULTS}")

# ---- 可选依赖自检（缺哪个，对应章节自动跳过） ----
HAS = {}
for mod, pip in [("pygame", "pygame"), ("imageio", "imageio + imageio-ffmpeg"),
                 ("llama_cpp", "llama-cpp-python"), ("torch", "torch"),
                 ("transformers", "transformers"), ("peft", "peft"),
                 ("vllm", "vllm"), ("bitsandbytes", "bitsandbytes")]:
    try:
        __import__(mod)
        HAS[mod] = True
    except Exception:
        HAS[mod] = False
print("[deps] " + "  ".join(f"{m}{'✓' if ok else '✗'}" for m, ok in HAS.items()))
print("[deps] 缺的可选依赖对应章节会打印「跳过」并继续")

## 1. 物理环境 GuyArena（零依赖）

60Hz 固定步长物理 / 5Hz 决策；8 个离散动作；对手内置 random（同频随机）与 dvd（屏保式弹跳+接触伤害）。数值平衡记录见原版 `DESIGN.md`。

In [ ]:
"""角色数值定义（v0）。

数值平衡目标（见 DESIGN.md §2.3）：
- Boxing/Random：近战 8 伤 / 0.5s 攻速，贴脸 DPS≈16
- DVD：120 血，弹跳移动，接触 5 伤 / 0.5s，被打出硬直会短暂停下
"""

GUY_TYPES = {
    "boxing": dict(
        kind="boxing", hp=100, speed=180.0, radius=16,
        attack_damage=8, attack_range=60.0, attack_cooldown=0.5,
        contact_damage=0,
    ),
    "random": dict(
        kind="random", hp=100, speed=180.0, radius=16,
        attack_damage=8, attack_range=60.0, attack_cooldown=0.5,
        contact_damage=0,
    ),
    "dvd": dict(
        kind="dvd", hp=120, speed=170.0, radius=18,
        attack_damage=0, attack_range=0.0, attack_cooldown=0.5,
        contact_damage=5, contact_cooldown=0.5,
    ),
}

In [ ]:
"""GuyArena v0 — headless 2D 竞技场物理核心（零依赖，纯 stdlib）。

- 60Hz 固定步长物理，5Hz 决策（每 DECISION_INTERVAL tick 查询一次 brain，期间重复动作）
- 8 个离散动作：NOOP/LEFT/RIGHT/UP/DOWN/ATTACK/BLOCK/SPECIAL（对应一个 Choice 问题）
- 对手内置策略：random（同频随机动作）/ dvd（DVD 屏保式弹跳 + 接触伤害，不吃动作）
- state dict 对齐 DESIGN.md §2.4 schema，目标 ≤300 token
"""
from __future__ import annotations

import math
import random
from enum import IntEnum


FPS = 60
DT = 1.0 / FPS
DECISION_INTERVAL = 12  # 60/12 = 5Hz 决策频率
ARENA_W, ARENA_H = 960, 540
MATCH_SECONDS = 60.0
MAX_TICKS = int(MATCH_SECONDS * FPS)

MELEE_WINDUP_TICKS = 6       # 出拳前摇 0.1s
HITSTUN_TICKS = 6            # 被打硬直 0.1s
BLOCK_TICKS = 24             # 格挡持续 0.4s
BLOCK_COOLDOWN = 1.0
BLOCK_DMG_RATIO = 0.3
DASH_TICKS = 9               # 冲刺持续 0.15s
DASH_COOLDOWN = 3.0
DASH_SPEED_MULT = 2.5
KNOCKBACK_HIT = 30.0
KNOCKBACK_BLOCKED = 10.0
KNOCKBACK_CONTACT = 40.0


class Action(IntEnum):
    NOOP = 0
    LEFT = 1
    RIGHT = 2
    UP = 3
    DOWN = 4
    ATTACK = 5
    BLOCK = 6
    SPECIAL = 7


ACTION_NAMES = ["NOOP", "LEFT", "RIGHT", "UP", "DOWN", "ATTACK", "BLOCK", "SPECIAL"]
MOVE_ACTIONS = {Action.LEFT: (-1, 0), Action.RIGHT: (1, 0), Action.UP: (0, -1), Action.DOWN: (0, 1)}


def _dist(a: "Guy", b: "Guy") -> float:
    return math.hypot(a.x - b.x, a.y - b.y)


class Guy:
    def __init__(self, kind: str, x: float, y: float):
        s = GUY_TYPES[kind]
        self.kind = kind
        self.x, self.y = float(x), float(y)
        self.vx = self.vy = 0.0
        self.max_hp = s["hp"]
        self.hp = s["hp"]
        self.radius = s["radius"]
        self.speed = s["speed"]
        self.attack_damage = s["attack_damage"]
        self.attack_range = s["attack_range"]
        self.attack_cooldown = s["attack_cooldown"]
        self.contact_damage = s.get("contact_damage", 0)
        self.contact_cooldown = s.get("contact_cooldown", 0.5)
        # 计时器
        self.cooldown = 0.0      # 攻击冷却（秒）
        self.windup = 0          # 出拳前摇（tick）
        self.block_ticks = 0
        self.block_cd = 0.0
        self.hitstun = 0
        self.dash_ticks = 0
        self.dash_cd = 0.0
        self.contact_cd = 0.0
        # 朝向（渲染/冲刺用）
        self.fx, self.fy = -1.0, 0.0

    @property
    def alive(self) -> bool:
        return self.hp > 0

    @property
    def blocking(self) -> bool:
        return self.block_ticks > 0


class GuyArena:
    """me 侧由外部 brain 控制；foe 侧为内置策略对手。"""

    def __init__(self, me_kind: str = "boxing", foe_kind: str = "dvd",
                 seed: int = 0, decision_interval: int = DECISION_INTERVAL,
                 w: float = ARENA_W, h: float = ARENA_H):
        if me_kind not in GUY_TYPES or foe_kind not in GUY_TYPES:
            raise ValueError(f"unknown guy kind: {me_kind} / {foe_kind}")
        self.me_kind = me_kind
        self.foe_kind = foe_kind
        self.seed = seed
        self.decision_interval = decision_interval
        self.w, self.h = w, h  # 场地尺寸（默认 960×540 = kev 训练分布）
        self.reset()

    def reset(self) -> dict:
        self.rng = random.Random(self.seed)
        self.tick = 0
        self.me = Guy(self.me_kind, self.w * 0.3, self.h / 2)
        self.foe = Guy(self.foe_kind, self.w * 0.7, self.h / 2)
        if self.foe_kind == "dvd":
            ang = self.rng.uniform(0, 2 * math.pi)
            self.foe.vx = math.cos(ang) * self.foe.speed
            self.foe.vy = math.sin(ang) * self.foe.speed
            self.foe.fx, self.foe.fy = self.foe.vx / self.foe.speed, self.foe.vy / self.foe.speed
        self.done = False
        self.outcome = None  # "me_win" | "foe_win" | "draw"
        self.last_foe_action = Action.NOOP
        self._foe_pending = Action.NOOP
        self.projectiles: list = []  # v0 占位（schema 保留）
        self.events: list = []
        return self.state()

    # ------------------------------------------------------------------ step
    def step(self, action: int):
        if self.done:
            return self.state(), 0.0, True, {"outcome": self.outcome}
        self.events = []
        action = Action(int(action))

        # 对手决策（与 brain 同频，保证公平）
        if self.tick % self.decision_interval == 0 and self.foe_kind == "random":
            self._foe_pending = Action(self.rng.randint(0, 7))
        foe_action = self._foe_pending if self.foe_kind == "random" else Action.NOOP
        self.last_foe_action = foe_action

        self._apply_action(self.me, action, self.foe)
        if self.foe_kind == "random":
            self._apply_action(self.foe, foe_action, self.me)

        self._integrate(self.me, is_dvd=False)
        self._integrate(self.foe, is_dvd=(self.foe_kind == "dvd"))
        self._resolve_melee(self.me, self.foe, side="me")
        self._resolve_melee(self.foe, self.me, side="foe")
        if self.foe_kind == "dvd":
            self._resolve_contact(self.foe, self.me)
        self._tick_timers(self.me)
        self._tick_timers(self.foe)

        self.tick += 1
        self._check_end()
        return self.state(), self._reward(), self.done, {"outcome": self.outcome}

    # -------------------------------------------------------------- internal
    def _apply_action(self, guy: Guy, action: Action, foe: Guy) -> None:
        if not guy.alive or guy.hitstun > 0:
            return
        if action in MOVE_ACTIONS:
            if guy.blocking:
                return
            dx, dy = MOVE_ACTIONS[action]
            guy.fx, guy.fy = float(dx), float(dy)
            if guy.dash_ticks == 0:
                guy.vx, guy.vy = dx * guy.speed, dy * guy.speed
        elif action == Action.NOOP:
            if guy.dash_ticks == 0:
                guy.vx = guy.vy = 0.0
        elif action == Action.ATTACK:
            if guy.cooldown <= 0 and not guy.blocking and guy.attack_damage > 0:
                guy.cooldown = guy.attack_cooldown
                guy.windup = MELEE_WINDUP_TICKS
                d = _dist(guy, foe)
                if d > 0:
                    guy.fx, guy.fy = (foe.x - guy.x) / d, (foe.y - guy.y) / d
        elif action == Action.BLOCK:
            if guy.block_cd <= 0 and not guy.blocking:
                guy.block_ticks = BLOCK_TICKS
                guy.block_cd = BLOCK_COOLDOWN
                guy.vx = guy.vy = 0.0
        elif action == Action.SPECIAL:
            if guy.dash_cd <= 0 and not guy.blocking:
                guy.dash_ticks = DASH_TICKS
                guy.dash_cd = DASH_COOLDOWN
                guy.vx, guy.vy = guy.fx * guy.speed * DASH_SPEED_MULT, guy.fy * guy.speed * DASH_SPEED_MULT

    def _integrate(self, guy: Guy, is_dvd: bool) -> None:
        if not guy.alive:
            return
        # 硬直：本 tick 不动，但 DVD 的弹跳速度保留（结束后继续弹）
        if guy.hitstun > 0 and not is_dvd:
            guy.vx = guy.vy = 0.0
        if guy.hitstun == 0 or is_dvd:
            guy.x += guy.vx * DT
            guy.y += guy.vy * DT
        r = guy.radius
        if guy.x < r:
            guy.x = r
            if is_dvd:
                guy.vx = abs(guy.vx)
        elif guy.x > self.w - r:
            guy.x = self.w - r
            if is_dvd:
                guy.vx = -abs(guy.vx)
        if guy.y < r:
            guy.y = r
            if is_dvd:
                guy.vy = abs(guy.vy)
        elif guy.y > self.h - r:
            guy.y = self.h - r
            if is_dvd:
                guy.vy = -abs(guy.vy)
        if is_dvd and (guy.vx or guy.vy):
            sp = math.hypot(guy.vx, guy.vy)
            if sp > 0:
                guy.fx, guy.fy = guy.vx / sp, guy.vy / sp

    def _resolve_melee(self, guy: Guy, foe: Guy, side: str) -> None:
        if guy.windup > 0:
            guy.windup -= 1
            if guy.windup == 0 and guy.alive and guy.attack_damage > 0:
                d = _dist(guy, foe)
                if d <= guy.attack_range and foe.alive:
                    blocked = foe.blocking
                    dmg = guy.attack_damage
                    knock = KNOCKBACK_HIT
                    if blocked:
                        dmg = max(1, int(round(dmg * BLOCK_DMG_RATIO)))
                        knock = KNOCKBACK_BLOCKED
                    else:
                        foe.hitstun = max(foe.hitstun, HITSTUN_TICKS)
                    foe.hp -= dmg
                    if d > 0:
                        foe.x += (foe.x - guy.x) / d * knock
                        foe.y += (foe.y - guy.y) / d * knock
                    self.events.append({"type": "hit", "side": side, "dmg": dmg, "blocked": blocked,
                                        "x": foe.x, "y": foe.y})
                else:
                    self.events.append({"type": "whiff", "side": side, "x": guy.x, "y": guy.y})

    def _resolve_contact(self, dvd: Guy, other: Guy) -> None:
        if not dvd.alive or not other.alive:
            return
        if dvd.contact_damage <= 0 or dvd.contact_cd > 0:
            return
        d = _dist(dvd, other)
        if d <= dvd.radius + other.radius + 2:
            dmg = dvd.contact_damage
            if other.blocking:
                dmg = max(1, int(round(dmg * BLOCK_DMG_RATIO)))
            else:
                other.hitstun = max(other.hitstun, 4)
            other.hp -= dmg
            if d > 0:
                other.x += (other.x - dvd.x) / d * KNOCKBACK_CONTACT
                other.y += (other.y - dvd.y) / d * KNOCKBACK_CONTACT
            dvd.contact_cd = dvd.contact_cooldown
            self.events.append({"type": "contact", "dmg": dmg, "x": other.x, "y": other.y})

    def _tick_timers(self, guy: Guy) -> None:
        guy.cooldown = max(0.0, guy.cooldown - DT)
        guy.block_cd = max(0.0, guy.block_cd - DT)
        guy.dash_cd = max(0.0, guy.dash_cd - DT)
        guy.contact_cd = max(0.0, guy.contact_cd - DT)
        guy.hitstun = max(0, guy.hitstun - 1)
        guy.block_ticks = max(0, guy.block_ticks - 1)
        if guy.dash_ticks > 0:
            guy.dash_ticks -= 1
            if guy.dash_ticks == 0 and guy.kind != "dvd":
                guy.vx = guy.vy = 0.0

    def _check_end(self) -> None:
        if not self.me.alive or not self.foe.alive:
            self.done = True
            self.outcome = "me_win" if self.me.alive else "foe_win"
        elif self.tick >= MAX_TICKS:
            self.done = True
            if self.me.hp > self.foe.hp:
                self.outcome = "me_win"
            elif self.foe.hp > self.me.hp:
                self.outcome = "foe_win"
            else:
                self.outcome = "draw"

    def _reward(self) -> float:
        if not self.done:
            return 0.0
        return {"me_win": 1.0, "foe_win": -1.0, "draw": 0.5}[self.outcome]

    # ----------------------------------------------------------------- views
    def state(self) -> dict:
        """对齐 DESIGN.md §2.4 的 state schema（补充 me.kind / me.range）。"""
        return {
            "t": round(self.tick * DT, 1),
            "me": {
                "kind": self.me.kind, "hp": self.me.hp,
                "x": round(self.me.x), "y": round(self.me.y),
                "vx": round(self.me.vx, 1), "vy": round(self.me.vy, 1),
                "cooldown": round(self.me.cooldown, 2), "blocking": self.me.blocking,
                "range": self.me.attack_range,
            },
            "foe": {
                "kind": self.foe.kind, "hp": self.foe.hp,
                "x": round(self.foe.x), "y": round(self.foe.y),
                "vx": round(self.foe.vx, 1), "vy": round(self.foe.vy, 1),
                "cooldown": round(self.foe.cooldown, 2),
            },
            "dist": round(_dist(self.me, self.foe), 1),
            "projectiles": [],
            "arena": {"w": self.w, "h": self.h},
            "last_foe_action": ACTION_NAMES[self.last_foe_action],
        }

    def snapshot(self) -> dict:
        """录像/渲染用的逐 tick 快照。"""
        def pack(g: Guy) -> dict:
            return {"x": round(g.x, 1), "y": round(g.y, 1), "hp": g.hp, "kind": g.kind,
                    "block": g.blocking, "windup": g.windup, "dash": g.dash_ticks > 0,
                    "fx": round(g.fx, 2), "fy": round(g.fy, 2)}
        return {"tick": self.tick, "me": pack(self.me), "foe": pack(self.foe),
                "events": self.events}

## 2. 大脑接口与内置大脑

所有决策端实现同一个接口 `decide(state) -> BrainDecision(action, probs)`——想换自己的大脑，实现这个接口即可。

In [ ]:
"""Brain 接口：所有决策端（随机/专家/SemIf/Kev）都实现 decide(state)。

返回 BrainDecision：
- action: 0-7（env.core.Action）
- probs: 可选，动作名 -> 概率（SemIf/Kev 会给出，用于校准分析与 HUD 展示）
"""
from __future__ import annotations

import random
from abc import ABC, abstractmethod
from dataclasses import dataclass, field


@dataclass
class BrainDecision:
    action: int
    probs: dict = field(default=None)


class Brain(ABC):
    name = "base"

    @abstractmethod
    def decide(self, state: dict) -> BrainDecision:
        ...


class RandomBrain(Brain):
    """纯随机基线：模拟"随机 guy"水平的下限。"""

    name = "random"

    def __init__(self, seed: int = 0):
        self.rng = random.Random(seed)

    def decide(self, state: dict) -> BrainDecision:
        return BrainDecision(self.rng.randint(0, 7))

In [ ]:
"""规则专家 bot（DESIGN.md §4.1 的有限规则白名单）。

用途：
1. M1 冒烟测试的环境平衡性基准（应能稳定赢 DVD/随机）
2. M3 模仿学习的数据采集器（那时由 data/collect.py 调用）
"""
from __future__ import annotations




def expert_action(state: dict) -> int:
    me, foe = state["me"], state["foe"]
    dist = state["dist"]
    atk_range = me.get("range", 60.0)

    # 对移动目标做线性外推拦截（dvd 弹道是确定性的），估计 0.5s 后的位置
    horizon = min(dist / 180.0, 0.5)
    tx = foe["x"] + foe.get("vx", 0.0) * horizon
    ty = foe["y"] + foe.get("vy", 0.0) * horizon
    dx = tx - me["x"]
    dy = ty - me["y"]
    dx_now = foe["x"] - me["x"]

    # 1) 低血且对面健康：拉开距离
    if me["hp"] < 30 and foe["hp"] > 60 and dist < 150:
        return int(Action.LEFT if dx_now > 0 else Action.RIGHT)

    # 2) 对面攻击冷却刚启动且贴脸（正在出拳前摇）：格挡
    if foe["kind"] != "dvd" and foe["cooldown"] > 0.35 and dist <= atk_range + 10:
        return int(Action.BLOCK)

    # 3) 不在射程：朝拦截点移动（x 优先，y 偏差太大时垂直修正）
    if dist > atk_range * 0.9:
        if abs(dy) > 100 and abs(dy) > abs(dx):
            return int(Action.DOWN if dy > 0 else Action.UP)
        return int(Action.RIGHT if dx > 0 else Action.LEFT)

    # 4) 进入射程：冷却好就出拳，否则等下一拍
    if me["cooldown"] <= 0:
        return int(Action.ATTACK)
    return int(Action.NOOP)


class ExpertBrain(Brain):
    name = "expert"

    def decide(self, state: dict) -> BrainDecision:
        return BrainDecision(expert_action(state))

## 3. 决策循环 · 录像 · 批量评估

`DecisionLoop`：60Hz 物理 + 每 `decision_interval` tick 查询一次大脑（间隔内重复动作）；`evaluate`：headless 批量跑 N 局输出胜率和 95% CI。

In [ ]:
"""录像：jsonl 格式（header + 逐 tick 快照 + 决策事件 + end）。

写入：DecisionLoop 传 record_path 即可。
回放：`python -m harness.replay <file.jsonl> [--speed 2]`
导帧：`python -m harness.replay <file.jsonl> --headless-frame out.png --at-tick 300`
"""
from __future__ import annotations

import json
import os
import sys


class ReplayWriter:
    def __init__(self, path: str, header: dict):
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        self.f = open(path, "w", encoding="utf-8")
        self._write({"type": "header", **header})

    def _write(self, obj: dict):
        self.f.write(json.dumps(obj, ensure_ascii=False) + "\n")

    def log_snapshot(self, snap: dict):
        self._write({"type": "snap", **snap})

    def log_decision(self, tick: int, t: float, action: str, probs: dict | None):
        self._write({"type": "decision", "tick": tick, "t": t, "action": action, "probs": probs})

    def close(self, outcome: str, ticks: int):
        self._write({"type": "end", "outcome": outcome, "ticks": ticks})
        self.f.close()


def load_replay(path: str):
    header, snaps, decisions, end = {}, [], {}, {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            t = obj.pop("type")
            if t == "header":
                header = obj
            elif t == "snap":
                snaps.append(obj)
            elif t == "decision":
                decisions[obj["tick"]] = obj
            elif t == "end":
                end = obj
    return header, snaps, decisions, end

In [ ]:
"""决策循环：60Hz 物理 + 5Hz  brain 查询，动作在间隔内重复。

用法：
  评估单局（headless）：python -m harness.loop --brain expert --foe dvd --seed 0
  录像：              python -m harness.loop --brain expert --foe dvd --record replays/x.jsonl
  实时观看：          python -m harness.loop --brain expert --foe dvd --live
"""
from __future__ import annotations

from dataclasses import dataclass, field





@dataclass
class MatchResult:
    outcome: str
    ticks: int
    hp_me: int
    hp_foe: int
    decisions: list = field(default_factory=list)


class DecisionLoop:
    def __init__(self, brain, foe: str = "dvd", seed: int = 0,
                 decision_interval: int = 12, record_path: str | None = None,
                 arena_w: float = 960, arena_h: float = 540):
        self.brain = brain
        self.arena = GuyArena(me_kind="boxing", foe_kind=foe,
                              seed=seed, decision_interval=decision_interval,
                              w=arena_w, h=arena_h)
        self.record_path = record_path
        self.foe = foe
        self.seed = seed

    def run_match(self) -> MatchResult:
        arena = self.arena
        state = arena.reset()
        recorder = None
        if self.record_path:
            ReplayWriter = globals()['ReplayWriter']  # 本 notebook 上文已定义
            recorder = ReplayWriter(self.record_path, header={
                "me": arena.me_kind, "foe": arena.foe_kind, "seed": self.seed,
                "brain": getattr(self.brain, "name", "?"), "fps": FPS,
                "decision_interval": arena.decision_interval,
                "arena": {"w": arena.w, "h": arena.h},
            })

        action = Action.NOOP
        decisions = []
        while not arena.done and arena.tick < MAX_TICKS + FPS * 2:
            if arena.tick % arena.decision_interval == 0:
                dec = self.brain.decide(state)
                action = Action(int(dec.action))
                decisions.append({"tick": arena.tick, "t": state["t"],
                                  "action": ACTION_NAMES[action], "probs": dec.probs})
                if recorder:
                    recorder.log_decision(arena.tick, state["t"], ACTION_NAMES[action], dec.probs)
            state, _, _, _ = arena.step(action)
            if recorder:
                recorder.log_snapshot(arena.snapshot())

        outcome = arena.outcome or "draw"
        if recorder:
            recorder.close(outcome, arena.tick)
        return MatchResult(outcome=outcome, ticks=arena.tick,
                           hp_me=arena.me.hp, hp_foe=arena.foe.hp,
                           decisions=decisions)

In [ ]:
import math


def evaluate(brain, foe: str, matches: int, seed: int = 0,
             decision_interval: int = None, arena_w: float = None, arena_h: float = None,
             verbose: bool = True) -> dict:
    """headless 批量评估：同一个 brain 实例跑 matches 局。"""
    decision_interval = decision_interval or RUNTIME["decision_interval"]
    arena_w = arena_w or ARENA["w"]
    arena_h = arena_h or ARENA["h"]
    wins = losses = draws = total_ticks = 0
    latencies = []
    for i in range(matches):
        loop = DecisionLoop(brain, foe=foe, seed=seed + i,
                            decision_interval=decision_interval,
                            arena_w=arena_w, arena_h=arena_h)
        r = loop.run_match()
        total_ticks += r.ticks
        lat = getattr(brain, "last_latency_ms", None)
        if lat is not None:
            latencies.append(lat)
        if r.outcome == "me_win":
            wins += 1
        elif r.outcome == "foe_win":
            losses += 1
        else:
            draws += 1
        if verbose:
            print(f"  [{i+1:>2}/{matches}] {r.outcome:<8} ticks={r.ticks:>5}"
                  + (f"  lat={lat:.0f}ms" if lat is not None else ""), flush=True)
    p = wins / matches
    ci = 1.96 * math.sqrt(p * (1 - p) / matches) if matches else 0.0
    return {"brain": getattr(brain, "name", "?"), "foe": foe, "matches": matches,
            "wins": wins, "losses": losses, "draws": draws,
            "win_rate": p, "ci95": ci, "avg_ticks": total_ticks / matches,
            "arena": f"{arena_w:.0f}x{arena_h:.0f}",
            "last_latency_ms": latencies[-1] if latencies else None}


def show_result(r: dict) -> None:
    line = (f"{r['brain']:>10} vs {r['foe']:<6} [{r['arena']}]  n={r['matches']:>3}  "
            f"win {r['wins']:>3} / loss {r['losses']:>3} / draw {r['draws']:>3}  "
            f"胜率 {r['win_rate']*100:5.1f}% ± {r['ci95']*100:4.1f}%  avg_ticks {r['avg_ticks']:.0f}")
    if r["last_latency_ms"] is not None:
        line += f"  单次决策 {r['last_latency_ms']:.0f}ms"
    print(line)

### 3.1 冒烟：规则专家 vs dvd（零依赖，几秒钟）

专家是本项目的"数据老师"（36k 训练集就是它自动对局打出来的），应稳定获胜。

In [ ]:
r = evaluate(ExpertBrain(), foe="dvd", matches=3)
show_result(r)
r = evaluate(RandomBrain(seed=0), foe="dvd", matches=3)
show_result(r)

## 4. 渲染 · 回放 · 视频导出（需要 pygame + imageio）

notebook 全程 headless（`SDL_VIDEODRIVER=dummy`）。`Renderer` 按录像 header 里的场地尺寸自适应（含窄屏 HUD 布局），`replay_to_video` 把 jsonl 录像转成 60fps mp4。

In [ ]:
import os as _os
_os.environ.setdefault("SDL_VIDEODRIVER", "dummy")  # 必须先于 pygame 初始化


In [ ]:
"""pygame 渲染器：实时观看 / 录像回放 / 导出帧。

- 深色背景 + 霓虹角色（me=红圆，dvd=蓝方块，random=绿圆）
- HUD：血条、冷却、当前决策动作与概率、结局横幅
- 依赖 pygame；headless 评估不需要本模块
"""
from __future__ import annotations

import math


BG = (18, 18, 24)
HUD_BG = (28, 28, 36)
GRID = (255, 255, 255)
ME_COLOR = (239, 83, 80)
DVD_COLOR = (66, 165, 245)
RANDOM_COLOR = (102, 187, 106)
TEXT = (230, 230, 235)
DIM = (150, 150, 160)
HIT_FLASH = (255, 214, 64)
BLOCK_COLOR = (77, 208, 225)
HUD_H = 96


def _kind_color(kind: str):
    return {"dvd": DVD_COLOR, "random": RANDOM_COLOR}.get(kind, ME_COLOR)


class Renderer:
    def __init__(self, title: str = "jev-guy", w: int = ARENA_W, h: int = ARENA_H):
        # w/h 默认 960x540（训练原尺寸）；场地迁移实验的录像按 replay header 里的
        # 实际尺寸构造，HUD 布局自适应：宽屏(>=760)血条居两侧+决策文本居中，
        # 窄屏决策文本移到第二行防止与血条重叠。
        import pygame
        self.pg = pygame
        pygame.init()
        self.w, self.h = int(w), int(h)
        self.screen = pygame.display.set_mode((self.w, self.h + HUD_H))
        pygame.display.set_caption(title)
        self.font = pygame.font.Font(None, 24)
        self.small_font = pygame.font.Font(None, 20)
        self.big_font = pygame.font.Font(None, 56)
        self.clock = pygame.time.Clock()
        self.particles: list = []

    # ------------------------------------------------------------------ draw
    def _draw_guy(self, g: dict, is_me: bool):
        pg = self.pg
        x, y = int(g["x"]), int(g["y"])
        color = _kind_color(g["kind"])
        if g["kind"] == "dvd":
            r = 18
            pg.draw.rect(self.screen, color, (x - r, y - r, r * 2, r * 2), border_radius=4)
        else:
            pg.draw.circle(self.screen, color, (x, y), 16)
        # 朝向短线
        pg.draw.line(self.screen, GRID, (x, y),
                     (int(x + g.get("fx", 0) * 26), int(y + g.get("fy", 0) * 26)), 2)
        if g.get("block"):
            pg.draw.circle(self.screen, BLOCK_COLOR, (x, y), 24, 3)
        if g.get("windup", 0) > 0:
            pg.draw.circle(self.screen, HIT_FLASH, (x, y), 22, 2)
        if g.get("dash"):
            pg.draw.circle(self.screen, GRID, (x, y), 28, 1)

    def _spawn_particles(self, events: list):
        for e in events:
            if e.get("type") in ("hit", "contact"):
                self.particles.append({"x": e["x"], "y": e["y"], "ttl": 14,
                                       "text": f"-{e['dmg']}"})

    def _draw_particles(self):
        pg = self.pg
        alive = []
        for p in self.particles:
            p["ttl"] -= 1
            if p["ttl"] > 0:
                radius = int(8 + (14 - p["ttl"]) * 1.5)
                pg.draw.circle(self.screen, HIT_FLASH, (int(p["x"]), int(p["y"])), radius, 2)
                img = self.font.render(p["text"], True, HIT_FLASH)
                self.screen.blit(img, (p["x"] + 12, p["y"] - 28 - (14 - p["ttl"])))
                alive.append(p)
        self.particles = alive

    def _draw_hud(self, snap: dict, info: dict):
        pg = self.pg
        top = self.h
        pg.draw.rect(self.screen, HUD_BG, (0, top, self.w, HUD_H))
        me, foe = snap["me"], snap["foe"]
        wide = self.w >= 760
        bw = 300 if wide else max(120, (self.w - 60) // 2)
        # 血条
        for i, (g, x, align) in enumerate([(me, 20, "left"), (foe, self.w - 20 - bw, "right")]):
            max_hp = 120 if g["kind"] == "dvd" else 100
            w = int(bw * max(0, g["hp"]) / max_hp)
            color = _kind_color(g["kind"])
            bx = x if align == "left" else x + (bw - w)
            pg.draw.rect(self.screen, (60, 60, 70), (x, top + 14, bw, 16), border_radius=3)
            pg.draw.rect(self.screen, color, (bx, top + 14, w, 16), border_radius=3)
            label = f"{'JEV GUY' if i == 0 else g['kind'].upper()}  {g['hp']}"
            img = self.font.render(label, True, TEXT)
            self.screen.blit(img, (x, top + 36) if align == "left"
                             else (x + bw - img.get_width(), top + 36))
        # 决策信息
        brain = info.get("brain", "?")
        action = info.get("action")
        probs = info.get("probs")
        tick = snap.get("tick", 0)
        t = tick / FPS
        line1 = f"brain={brain}  t={t:4.1f}s  action={action or '-'}"
        if wide:
            self.screen.blit(self.font.render(line1, True, TEXT), (330, top + 10))
            if probs:
                top3 = sorted(probs.items(), key=lambda kv: -kv[1])[:3]
                line2 = "  ".join(f"{k}:{v:.2f}" for k, v in top3)
                self.screen.blit(self.font.render(line2, True, DIM), (330, top + 34))
        else:
            self.screen.blit(self.small_font.render(line1, True, TEXT), (20, top + 58))
            if probs:
                top3 = sorted(probs.items(), key=lambda kv: -kv[1])[:3]
                line2 = "  ".join(f"{k}:{v:.2f}" for k, v in top3)
                self.screen.blit(self.small_font.render(line2, True, DIM), (20, top + 76))
        # 结局横幅
        outcome = info.get("outcome")
        if outcome:
            text = {"me_win": "JEV GUY WINS", "foe_win": "FOE WINS", "draw": "DRAW"}.get(outcome, outcome)
            img = self.big_font.render(text, True, HIT_FLASH)
            self.screen.blit(img, ((self.w - img.get_width()) // 2,
                                   (self.h - img.get_height()) // 2))

    def draw_snapshot(self, snap: dict, info: dict):
        self._spawn_particles(snap.get("events", []))
        self.screen.fill(BG)
        self.pg.draw.rect(self.screen, (40, 40, 52), (0, 0, self.w, self.h), 1)
        self._draw_particles()
        self._draw_guy(snap["me"], is_me=True)
        self._draw_guy(snap["foe"], is_me=False)
        self._draw_hud(snap, info)
        self.pg.display.flip()

    def save(self, path: str):
        self.pg.image.save(self.screen, path)

    # ------------------------------------------------------------------ play
    def play(self, snaps: list, decisions: dict, end: dict, info: dict, speed: float = 1.0):
        running = True
        i = 0
        last_dec = None
        while running and i < len(snaps):
            for ev in self.pg.event.get():
                if ev.type == self.pg.QUIT:
                    running = False
                    break
            snap = snaps[i]
            dec = decisions.get(snap["tick"])
            if dec is not None:
                last_dec = dec
            show_end = end if i >= len(snaps) - 1 else {}
            self.draw_snapshot(snap, {**info,
                                      "action": last_dec["action"] if last_dec else None,
                                      "probs": last_dec.get("probs") if last_dec else None,
                                      **show_end})
            self.clock.tick(int(FPS * speed))
            i += 1
        # 结束后停留 2s
        if running:
            self.pg.time.wait(2000)
        self.pg.quit()


def live_play(loop):
    """实时渲染一局：loop 是 harness.loop.DecisionLoop。"""
    ACTION_NAMES, Action = globals()['ACTION_NAMES'], globals()['Action']

    arena = loop.arena
    r = Renderer(title=f"jev-guy live: {loop.brain.name} vs {loop.foe}",
                 w=arena.w, h=arena.h)
    state = arena.reset()
    action = Action.NOOP
    running = True
    last_dec = None
    while running and not arena.done:
        for ev in r.pg.event.get():
            if ev.type == r.pg.QUIT:
                running = False
                break
        if arena.tick % arena.decision_interval == 0:
            dec = loop.brain.decide(state)
            action = Action(int(dec.action))
            last_dec = dec
        state, _, _, _ = arena.step(action)
        r._spawn_particles(arena.events)
        r.screen.fill(BG)
        r.pg.draw.rect(r.screen, (40, 40, 52), (0, 0, r.w, r.h), 1)
        r._draw_particles()
        snap = arena.snapshot()
        r._draw_guy(snap["me"], True)
        r._draw_guy(snap["foe"], False)
        r._draw_hud(snap, {"brain": loop.brain.name,
                           "action": ACTION_NAMES[action],
                           "probs": getattr(last_dec, "probs", None),
                           "outcome": arena.outcome if arena.done else None})
        r.pg.display.flip()
        r.clock.tick(FPS)
    r.pg.time.wait(2500)
    r.pg.quit()

In [ ]:
def replay_to_video(replay_path, out_path, fps: int = 60, every: int = 1, quality: int = 7):
    """录像 jsonl → mp4（结局横幅在最后 ~2s 持续显示）。"""
    import imageio.v2 as imageio
    import pygame

    header, snaps, decisions, end = load_replay(str(replay_path))
    arena_hdr = header.get("arena") or {}
    r = Renderer(title="replay", w=arena_hdr.get("w", 960), h=arena_hdr.get("h", 540))

    def latest_decision(tick):
        best = None
        for t, d in decisions.items():
            if t <= tick and (best is None or t > best[0]):
                best = (t, d)
        return best[1] if best else None

    n = len(snaps)
    banner_from = max(0, n - fps * 2)
    w = imageio.get_writer(str(out_path), fps=fps, codec="libx264", quality=quality)
    for i, snap in enumerate(snaps):
        if i % every:
            continue
        dec = latest_decision(snap["tick"])
        r.draw_snapshot(snap, {"brain": header.get("brain", "?"),
                               "action": dec["action"] if dec else None,
                               "probs": dec.get("probs") if dec else None,
                               "outcome": end.get("outcome") if i >= banner_from else None})
        w.append_data(pygame.surfarray.array3d(r.screen).transpose(1, 0, 2))
    w.close()
    pygame.quit()
    print(f"video -> {out_path} ({(n + every - 1) // every} frames, {fps}fps)")
    return out_path

### 4.1 演示：专家打一局 → 录像 → 转 mp4 → 抽帧展示

产物写入 `results_root/replays/`。

In [ ]:
if not (HAS["pygame"] and HAS["imageio"]):
    print("[skip] 缺 pygame/imageio，跳过视频演示")
else:
    rec = RESULTS / "replays" / f"expert_demo_s{DEMO['record_seed']}.jsonl"
    loop = DecisionLoop(ExpertBrain(), foe="dvd", seed=DEMO["record_seed"], record_path=str(rec))
    result = loop.run_match()
    print(f"outcome={result.outcome}  ticks={result.ticks}  hp={result.hp_me}-{result.hp_foe}")
    mp4 = replay_to_video(rec, str(rec).replace(".jsonl", ".mp4"))

    # 抽最后一帧展示
    from IPython.display import Image as IPyImage
    from IPython.display import display
    _os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
    header, snaps, decisions, end = load_replay(str(rec))
    r = Renderer(w=header["arena"]["w"], h=header["arena"]["h"])
    last = snaps[-1]
    dec = decisions.get(last["tick"]) or list(decisions.values())[-1]
    r.draw_snapshot(last, {"brain": "expert", "action": dec["action"],
                           "probs": dec.get("probs"), "outcome": end.get("outcome")})
    frame = RESULTS / "replays" / "expert_demo_lastframe.png"
    r.save(str(frame))
    display(IPyImage(filename=str(frame)))

## 5. SemIf 零训练大脑（可选：llama-cpp-python + GGUF）

Jev 范式的零训练形态：规则文本 + 符号化方位 hints + state JSON → llama.cpp 一次前向（**不解码**）→ 只取 8 个字母位 logits → softmax。

⚠️ 训练分布声明：prompt 里的 `RULES_TEXT` 写死 960×540（36k 数据全在该尺寸采集），`legacy_arena=True` 可强制"无论实际场地都按 960 说话"的对照臂。

In [ ]:
"""SemIf 零训练决策脑（M2）。

机制（对齐 github.com/TheoLeeCJ/SemIf 与 DESIGN.md §3.1）：
1. 规则文本 + state JSON + 8 个动作候选拼成 prompt（chat 模板，assistant 预填 "Action:"）
2. 一次前向（llama.cpp eval），**不做任何自回归解码**
3. 只取 8 个选项字母位的 logits → softmax(温度 T) → 概率 + argmax 动作

为什么用 llama.cpp + GGUF 而不是 transformers bf16：
- RTX 4050 Laptop 6GB 显存装不下 4B bf16（~8.8GB），Q4_K_M 仅 ~2.6GB
- llama.cpp 的 logits_all 能直接拿最后一个位置的完整 logits，正是 SemIf 需要的
- 附带收获：E4 量化实验时 Q4/Q5 对比即换文件

延迟说明：单次决策 ≈ 一次 300 token prefill。4050 上预计 0.3–1s，
所以默认决策间隔建议 12–24 tick（2.5–5Hz），由 harness 的 --decision-interval 控制。
"""
from __future__ import annotations

import json
import os

import numpy as np



RULES_TEXT = """You are the controller of a boxing fighter (red) in a 2D arena game.
Read the game state JSON and choose the best action for this moment.

Coordinates (IMPORTANT):
- x increases to the RIGHT: 0 = left wall, 960 = right wall.
- y increases DOWNWARD: 0 = top, 540 = bottom.
- So: enemy.x < your.x means the enemy is to your LEFT -> move LEFT to approach.
- Moving into a wall does nothing (position is clamped). Do NOT dash (SPECIAL) toward a wall.

Game rules:
- Your melee ATTACK hits only if dist <= your range (60 px). ATTACK has 0.5s cooldown.
- If the enemy has just started an attack (their cooldown is high) and you are close, BLOCK cuts damage to 30%.
- To hit the enemy you must first get in range: move toward them. Use their velocity (vx, vy) to intercept, not chase.
- SPECIAL is a dash (2.5x speed burst) toward your CURRENT facing, useful to close distance fast. Cooldown 3s.
- If your hp is low and the enemy is much healthier, retreat to buy time.
- "dvd" enemies hurt you on contact and bounce off walls; they never stop moving.

Actions (answer with ONE letter):
A. NOOP - stay still
B. LEFT - move left (decrease x)
C. RIGHT - move right (increase x)
D. UP - move up (decrease y)
E. DOWN - move down (increase y)
F. ATTACK - punch (needs dist <= 60)
G. BLOCK - reduce incoming damage for 0.4s
H. SPECIAL - dash toward your current facing direction
"""

LETTERS = "ABCDEFGH"

# ------------------------------------------------------------------ 场地感知（2026-10-07）
# ⚠️ 训练分布声明：kev 微调模型的 36k 训练数据全部在 960×540 场地上采集，
# 且 prompt 中的 RULES_TEXT 写死了 960/540 两个数字、墙阈值写死 880/80。
# 因此：960×540 时 RULES_TEXT 必须逐字节保持原版（训推一致的生命线）；
# 其他尺寸只是"让 prompt 不说谎"的零样本迁移尝试，行为保真度以实测为准。
# legacy_arena=True 可强制回到"无论实际场地都按 960×540 说话"的旧行为（前后对照臂）。
_ORIGINAL_W, _ORIGINAL_H = 960, 540
_WALL_MARGIN = 80  # 原版阈值的来源：960-80=880 / 0+80=80


def _arena_of(state: dict, legacy_arena: bool) -> tuple[float, float]:
    if legacy_arena:
        return _ORIGINAL_W, _ORIGINAL_H
    a = state.get("arena") or {}
    return float(a.get("w", _ORIGINAL_W)), float(a.get("h", _ORIGINAL_H))


def _rules_text(w: float, h: float) -> str:
    """960×540 时返回冻结原版（逐字节一致）；其他尺寸只改两个数字。"""
    if int(w) == _ORIGINAL_W and int(h) == _ORIGINAL_H:
        return RULES_TEXT
    return (RULES_TEXT
            .replace("960 = right wall", f"{int(w)} = right wall")
            .replace("540 = bottom", f"{int(h)} = bottom"))


def _direction_hints(state: dict, legacy_arena: bool = False) -> str:
    """把原始坐标换算成方向语言——4B 模型的空间推理不可靠，
    预计算的方位提示比让它自己算 x 差值靠谱得多（M2.1 prompt 迭代）。
    墙阈值随场地尺寸变化（margin 恒 80px，960 宽时即原版的 880/80）。"""
    aw, ah = _arena_of(state, legacy_arena)
    me, foe = state["me"], state["foe"]
    dx = foe["x"] - me["x"]
    dy = foe["y"] - me["y"]
    dist = state["dist"]
    rng = me.get("range", 60.0)

    horiz = f"{abs(dx):.0f} px to your {'LEFT' if dx < 0 else 'RIGHT'}" if abs(dx) > 20 else "almost level with you"
    vert = f"{abs(dy):.0f} px {'ABOVE' if dy < 0 else 'BELOW'} you" if abs(dy) > 20 else "same height"
    lines = [f"Situation: the enemy is {horiz}, {vert}."]

    if dist <= rng:
        lines.append(f"dist={dist:.0f} <= your range {rng:.0f} -> ATTACK can hit NOW.")
    else:
        lines.append(f"dist={dist:.0f} > your range {rng:.0f} -> you cannot hit yet; close the gap first.")

    if me["x"] > aw - _WALL_MARGIN:
        lines.append("You are at the RIGHT wall: RIGHT/SPECIAL do nothing useful, go LEFT.")
    elif me["x"] < _WALL_MARGIN:
        lines.append("You are at the LEFT wall: LEFT does nothing useful, go RIGHT.")

    if me.get("cooldown", 0) > 0:
        lines.append(f"Your attack is on cooldown ({me['cooldown']:.1f}s left) — ATTACK would waste the turn.")
    else:
        lines.append("Your attack is ready (cooldown 0).")

    fvx, fvy = foe.get("vx", 0.0), foe.get("vy", 0.0)
    if abs(fvx) > 1 or abs(fvy) > 1:
        lines.append(f"Enemy velocity=({fvx:.0f},{fvy:.0f}) — aim where it is going, not where it is.")
    return "\n".join(lines)


def build_prompt(state: dict, legacy_arena: bool = False) -> str:
    """legacy_arena=False（默认）：按 state["arena"] 的真实尺寸生成规则与提示；
    legacy_arena=True：强制按 960×540 说话（旧行为，用于"prompt 撒谎"对照臂）。"""
    aw, ah = _arena_of(state, legacy_arena)
    state_json = json.dumps(state, ensure_ascii=False, separators=(",", ":"))
    hints = _direction_hints(state, legacy_arena)
    return (
        f"<|im_start|>system\n{_rules_text(aw, ah)}<|im_end|>\n"
        f"<|im_start|>user\nState: {state_json}\n{hints}\n"
        f"Which action is best right now? Answer with one letter (A-H).<|im_end|>\n"
        f"<|im_start|>assistant\nAction:"
    )


def _ensure_cuda_dll_dirs() -> None:
    """把 pip 版 nvidia 运行时（nvidia-cuda-runtime-cu12 / nvidia-cublas-cu12）的
    bin 目录注册进 DLL 搜索路径。llama.cpp cu124 的 ggml-cuda.dll 依赖
    cudart64_12/cublas64_12，wheel 不打包，由这里补齐（Windows 专属问题）。"""
    import glob
    import site
    import sys

    if sys.platform != "win32":
        return
    candidates = []
    for root in site.getsitepackages() + [site.getusersitepackages()]:
        candidates += glob.glob(os.path.join(root, "nvidia", "*", "bin"))
    for d in candidates:
        if os.path.isdir(d):
            os.environ["PATH"] = d + os.pathsep + os.environ.get("PATH", "")
            try:
                os.add_dll_directory(d)
            except (AttributeError, OSError):
                pass


class SemIfBrain(Brain):
    name = "semif-qwen35-4b-q4"

    def __init__(self, model_path: str, n_ctx: int = 1024, temperature: float = 1.0,
                 n_gpu_layers: int = -1, seed: int = 0, legacy_arena: bool = False):
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"模型文件不存在: {model_path}")
        _ensure_cuda_dll_dirs()
        from llama_cpp import Llama  # 延迟导入，未装 llama-cpp 时不影响其他 brain

        self.temperature = temperature
        self.legacy_arena = legacy_arena
        self.llm = Llama(model_path=model_path, n_ctx=n_ctx, logits_all=True,
                         n_gpu_layers=n_gpu_layers, seed=seed, verbose=False)
        # 候选字母的 token id（兼容 " A"/"A" 两种切分，取每字母的最高者）
        self._letter_id_sets: list[list[int]] = []
        for ch in LETTERS:
            ids = set()
            for variant in (" " + ch, ch):
                toks = self.llm.tokenize(variant.encode("utf-8"), add_bos=False)
                if toks:
                    ids.add(toks[-1])
            self._letter_id_sets.append(sorted(ids))
        self._last_latency_ms = None

    # ------------------------------------------------------------------ API
    def decide(self, state: dict) -> BrainDecision:
        import time

        prompt = build_prompt(state, legacy_arena=self.legacy_arena)
        tokens = self.llm.tokenize(prompt.encode("utf-8"), add_bos=True)
        if len(tokens) > self.llm.n_ctx() - 2:
            raise ValueError(f"prompt 超长: {len(tokens)} tokens > n_ctx={self.llm.n_ctx()}")

        t0 = time.perf_counter()
        # 每次决策全量重算（~650ms）。曾尝试手动前缀缓存（公共前缀 KV 复用），
        # 但 Qwen3.5 是 Gated DeltaNet 混合架构：线性注意力层的状态是递推的，
        # KV 截断无法回滚它 → llama_decode 直接报错。混合架构的 prefix cache
        # 需要框架级支持（留给 E3 用 vLLM 做，见 DESIGN.md）。
        self.llm.reset()
        self.llm.eval(tokens)
        # 注意：llama-cpp 的 scores 缓冲区是 (n_ctx, n_vocab) 预分配，
        # 未评估的行全为 0——必须按"最后一个已评估 token"的位置取，不能用 [-1]
        logits = np.asarray(self.llm.scores[len(tokens) - 1], dtype=np.float64)
        self._last_latency_ms = (time.perf_counter() - t0) * 1000

        opt = np.array([max(logits[i] for i in ids) for ids in self._letter_id_sets])
        z = opt / max(self.temperature, 1e-6)
        z = z - z.max()
        p = np.exp(z)
        p /= p.sum()
        action = int(p.argmax())
        return BrainDecision(action=action,
                             probs={ACTION_NAMES[i]: float(p[i]) for i in range(len(ACTION_NAMES))})

    @property
    def last_latency_ms(self):
        return self._last_latency_ms

### 5.1 SemIf 冒烟 + 评估（需配置 `paths.gguf_q4`）

In [ ]:
if not HAS["llama_cpp"]:
    print("[skip] 未安装 llama-cpp-python")
elif not PATHS.get("gguf_q4") or not pathlib.Path(PATHS["gguf_q4"]).exists():
    print("[skip] 未配置 paths.gguf_q4（或文件不存在）——见 index.md 下载说明")
else:
    semif = SemIfBrain(model_path=str(PATHS["gguf_q4"]),
                       temperature=RUNTIME["temperature"], n_gpu_layers=-1)

    # 两个手工场景冒烟：贴脸该出拳，远距该追击
    def _s(me_x, foe_x, dist, cd=0.0):
        return {"t": 10.0, "me": {"kind": "boxing", "hp": 100, "x": me_x, "y": 270,
                                  "vx": 0.0, "vy": 0.0, "cooldown": cd, "blocking": False,
                                  "range": 60.0},
                "foe": {"kind": "dvd", "hp": 120, "x": foe_x, "y": 270,
                        "vx": 120.0, "vy": 0.0, "cooldown": 0.0},
                "dist": dist, "projectiles": [], "arena": {"w": 960, "h": 540},
                "last_foe_action": "NOOP"}

    for title, st in [("贴脸且冷却好（应 ATTACK）", _s(430, 480, 52.0)),
                      ("远距离对峙（应追击）", _s(100, 800, 705.0))]:
        dec = semif.decide(st)
        top3 = sorted(dec.probs.items(), key=lambda kv: -kv[1])[:3]
        print(f"[{title}] " + "  ".join(f"{k}={v:.3f}" for k, v in top3)
              + f"  延迟 {semif.last_latency_ms:.0f}ms")

    show_result(evaluate(semif, foe="dvd", matches=DEMO["eval_matches"]))

## 6. Kev 微调大脑（可选：transformers + peft + bf16 基座 + adapter）

与 SemIf 同构（同一个 `build_prompt`、同一条"答案位 hidden → lm_head 一行 → 8 字母位 logits"路径），保证训推一致。
adapter 可用开源包自带的成品（`kev_v1/ckpt-step2000`，96.7% 战绩），也可用第 8 节自己训。

In [ ]:
"""Kev 微调决策脑（M3-3）：base bf16 + LoRA adapter，transformers 加载。

与 SemIfBrain 同构（Jev 范式：一次前向、只取 8 字母位 logits、不做生成），
与 data/train_kev.py 训练侧严格同路径（同一个 build_prompt、同一条
decoder→last hidden→lm_head→8 字母位 的取 logit 路线），保证训推一致。

为什么不用 llama.cpp 加载 adapter：
- llama.cpp 对 LoRA adapter 的支持要求先 merge 或转 GGUF，多一道工序；
  transformers 直接 base+adapter 加载，与训练产物零转换。
- 128GB 统一内存装 bf16 4B（8.8GB）毫无压力，不需要 Q4 量化省显存。
- E4 量化实验时再走 merge→GGUF 路线（那时量化本身就是实验对象）。
"""
from __future__ import annotations

import json
import os
import time

import numpy as np




def _pick_device(name: str) -> str:
    import torch
    if name != "auto":
        return name
    if torch.cuda.is_available():
        return "cuda"
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return "mps"
    return "cpu"


class KevBrain(Brain):
    name = "kev-qwen35-4b-lora"

    def __init__(self, base_path: str, adapter_path: str, device: str = "auto",
                 temperature: float = 1.0, seed: int = 0, legacy_arena: bool = False):
        if not os.path.isdir(base_path):
            raise FileNotFoundError(f"base 模型目录不存在: {base_path}")
        if not os.path.isdir(adapter_path):
            raise FileNotFoundError(f"adapter 目录不存在: {adapter_path}")
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from peft import PeftModel

        self.torch = torch
        self.device = _pick_device(device)
        self.temperature = temperature
        self.legacy_arena = legacy_arena

        # tokenizer 优先从 adapter 目录读（正式训练结尾会存），
        # 快照目录（ckpt-stepN）只有 adapter 权重 → 回退到 base 目录
        try:
            self.tok = AutoTokenizer.from_pretrained(adapter_path)
        except (OSError, ValueError):
            self.tok = AutoTokenizer.from_pretrained(base_path)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        base = AutoModelForCausalLM.from_pretrained(
            base_path, torch_dtype=torch.bfloat16,
            attn_implementation="sdpa", trust_remote_code=True)
        self.model = PeftModel.from_pretrained(base, adapter_path)
        self.model.to(self.device)
        self.model.eval()

        # letter_ids：优先读训练时落的 letters.json（训推对齐的单一事实源），
        # 没有就按训练侧同样的方法现算（" A" 单 token 变体）
        letters_json = os.path.join(adapter_path, "letters.json")
        if os.path.exists(letters_json):
            with open(letters_json) as f:
                self.letter_ids = json.load(f)["letter_ids"]
        else:
            self.letter_ids = []
            for ch in LETTERS:
                ids = self.tok(" " + ch, add_special_tokens=False)["input_ids"]
                assert len(ids) == 1, f'" {ch}" 不是单 token: {ids}'
                self.letter_ids.append(ids[0])

        self._last_latency_ms = None

    # ------------------------------------------------------------------ API
    def decide(self, state: dict) -> BrainDecision:
        torch = self.torch
        prompt = build_prompt(state, legacy_arena=self.legacy_arena)  # 与训练完全同一个 prompt 构造函数
        enc = self.tok([prompt], return_tensors="pt").to(self.device)

        t0 = time.perf_counter()
        with torch.no_grad():
            # 与 train_kev.last_position_logits 同路径：decoder → 答案位 hidden
            # → lm_head 只算一行 → 8 字母位 logits（不算全序列 (B,T,V) logits）
            dec = self.model.get_decoder()
            hs = dec(**enc).last_hidden_state
            last_h = hs[:, -1, :]
            logits = self.model.get_output_embeddings()(last_h)[0, self.letter_ids]
            opt = logits.float().cpu().numpy().astype(np.float64)
        self._last_latency_ms = (time.perf_counter() - t0) * 1000

        z = opt / max(self.temperature, 1e-6)
        z = z - z.max()
        p = np.exp(z)
        p /= p.sum()
        action = int(p.argmax())
        return BrainDecision(action=action,
                             probs={ACTION_NAMES[i]: float(p[i]) for i in range(len(ACTION_NAMES))})

    @property
    def last_latency_ms(self):
        return self._last_latency_ms

### 6.1 Kev 评估 + 录像（需配置 `paths.base_bf16` 与 `paths.kev_adapter`）

In [ ]:
if not (HAS["torch"] and HAS["transformers"] and HAS["peft"]):
    print("[skip] 未安装 torch/transformers/peft")
elif not (PATHS.get("base_bf16") and PATHS.get("kev_adapter")):
    print("[skip] 未配置 paths.base_bf16 / paths.kev_adapter——见 index.md 下载说明")
else:
    kev = KevBrain(base_path=str(PATHS["base_bf16"]), adapter_path=str(PATHS["kev_adapter"]),
                   device=RUNTIME["device"], temperature=RUNTIME["temperature"])
    print(f"kev loaded (device={kev.device})")
    show_result(evaluate(kev, foe="dvd", matches=DEMO["eval_matches"]))

    # 录一局对战 → 转 mp4
    rec = RESULTS / "replays" / f"kev_demo_s{DEMO['record_seed']}.jsonl"
    loop = DecisionLoop(kev, foe="dvd", seed=DEMO["record_seed"], record_path=str(rec))
    result = loop.run_match()
    print(f"kev 对局: outcome={result.outcome}  ticks={result.ticks}  hp={result.hp_me}-{result.hp_foe}")
    if HAS["pygame"] and HAS["imageio"]:
        replay_to_video(rec, str(rec).replace(".jsonl", ".mp4"))

## 7. 数据采集（可选，纯 CPU，较慢）

专家 bot 自动对局 → 模仿学习样本（dvd 40% / random 40% / 噪声镜像 20%；5px 网格去重；类别限流）。
36k 正式数据（300 局）需要几十分钟，演示先跑几局看格式。

In [ ]:
"""M3 数据采集器：专家 bot 自动对局 → 模仿学习样本集。

样本格式（jsonl 每行一条）：
{
  "state":  {...},            # env.core.GuyArena.state()（与 M2 模型输入一致）
  "hints":  "...",            # brain.semif._direction_hints 的符号化方位提示
  "action": 5,                # 专家标签动作（env.core.Action）
  "action_name": "ATTACK",
  "foe": "dvd", "seed": 12, "tick": 1206
}

设计要点（对齐 DESIGN.md §4）：
- 对手混合：dvd 40% / random 40% / 带噪声专家镜像 20%（镜像局也由专家打标签）
- 每 6 tick 采样；相邻帧"近似重复"去重（位置 5px 网格 + 冷却桶 + 同动作）
- 类别限流：NOOP/移动类设上限，ATTACK/BLOCK/SPECIAL 不限（稀有类全收）
- 纯 CPU，不吃显卡；200 局 ≈ 12 万原始帧 → 限流后 ~5 万样本

用法：
  python -m data.collect --matches 200 --out data/dataset/expert_v1.jsonl
  python -m data.collect --matches 3 --out data/dataset/smoke.jsonl   # 冒烟
"""
from __future__ import annotations

import json
import random
from collections import Counter



SAMPLE_EVERY = 6
FOE_MIX = ["dvd", "dvd", "random", "random", "mirror"]  # 40/40/20
MIRROR_NOISE = 0.08

# 类别限流（None = 不限）
CLASS_CAP = {
    "NOOP": 3000, "LEFT": 9000, "RIGHT": 9000, "UP": 4000, "DOWN": 4000,
    "ATTACK": None, "BLOCK": None, "SPECIAL": None,
}


def noisy_expert(state: dict, rng: random.Random) -> int:
    if rng.random() < MIRROR_NOISE:
        return rng.randint(0, 7)
    return expert_action(state)


def _dedup_key(state: dict, action: int) -> tuple:
    me, foe = state["me"], state["foe"]
    return (int(me["x"] // 5), int(me["y"] // 5),
            int(foe["x"] // 5), int(foe["y"] // 5),
            round(me["cooldown"] * 2) / 2, round(foe["cooldown"] * 2) / 2,
            action)


def collect_match(foe: str, seed: int, rng: random.Random) -> list:
    arena = GuyArena(me_kind="boxing", foe_kind=("dvd" if foe == "mirror" else foe), seed=seed)
    state = arena.reset()
    action = 0
    samples = []
    last_key = None
    while not arena.done:
        if arena.tick % arena.decision_interval == 0:
            action = noisy_expert(state, rng) if foe == "mirror" else expert_action(state)
        if arena.tick % SAMPLE_EVERY == 0:
            label = expert_action(state)
            key = _dedup_key(state, label)
            if key != last_key:
                samples.append({
                    "state": state,
                    "hints": _direction_hints(state),
                    "action": label,
                    "action_name": ACTION_NAMES[label],
                    "foe": foe, "seed": seed, "tick": arena.tick,
                })
                last_key = key
        state, _, _, _ = arena.step(action)
    return samples

In [ ]:
import json
from collections import Counter


def collect_dataset(out_path, matches: int, seed: int = 100) -> None:
    rng = __import__("random").Random(seed)
    kept, capped = 0, 0
    class_count: Counter = Counter()
    with open(out_path, "w", encoding="utf-8") as f:
        for i in range(matches):
            foe = FOE_MIX[i % len(FOE_MIX)]
            for s in collect_match(foe, seed=seed + i, rng=rng):
                cap = CLASS_CAP[s["action_name"]]
                if cap is not None and class_count[s["action_name"]] >= cap:
                    capped += 1
                    continue
                class_count[s["action_name"]] += 1
                f.write(json.dumps(s, ensure_ascii=False) + "\n")
                kept += 1
    print(f"完成: {matches} 局 -> {kept} 样本（限流丢弃 {capped}）-> {out_path}")
    total = sum(class_count.values())
    for name, n in class_count.most_common():
        print(f"  {name:>8}: {n:>6}  ({n/total*100:4.1f}%)")


# 演示：3 局小样本（正式采集把 matches 调到 300，输出到 RESULTS/dataset/）
collect_dataset(RESULTS / "dataset" / "smoke.jsonl", matches=3)

## 8. 训练 Kev（可选，需 GPU + bf16 基座 + 数据集）

Jev 范式的训练：**损失只落在答案位的 8 个字母 logits 上**（交叉熵），绕过 248,320 词表的全量 logits——这是"决策模型训练"与"语言模型训练"的本质区别，也是显存省钱大头。

- CUDA：默认 NF4 QLoRA + 梯度检查点 + PagedAdamW8bit（24GB 卡 ~7s/step）；
- Mac MPS / CPU：自动降级 bf16 + adamw_torch；
- `--save-every` 快照逻辑已内置（每 500 步存 adapter，防实例到点被杀——96.7% 的成品就是这么救回来的）；
- E1 对照开关：`no_ckpt / no_amp / quant="none" / optim="adamw_torch_fused"`。

In [ ]:
"""M3 训练脚本：Kev 风格 QLoRA 微调（Jev 范式 = 只在 8 个选项位算交叉熵）。

核心设计（与 brain/semif.py 推理完全同构，保证训推一致）：
- prompt 用同一个 build_prompt(state)（规则文本 + hints + state JSON）
- 只取答案位（prompt 最后一个 token 位置）的 hidden state → lm_head → 8 个字母位 logits
- loss = 这 8 个 logits 上的交叉熵。**不算全词表 LM loss，不做生成**——
  这正是"决策模型"和"语言模型"训练的区别，也是显存省钱的来源

显存优化（E1 账本实验对象，DESIGN.md §5）：
- bnb NF4 4bit 基座（6GB 卡的硬前提）+ LoRA r16 + 梯度检查点 + PagedAdamW8bit
- 每 log 窗口打印 peak allocated/reserved（torch.cuda.max_memory_*），结束落账本 json

跨平台（2026-10-05 Mac 移植）：
- --device auto：cuda > mps > cpu。CUDA 路径与原版逐字节一致（AutoDL 继续用）
- MPS/CPU：无 NF4（bitsandbytes 不可靠）→ bf16 直接加载（M5 Max 128GB 统一内存装得下）；
  optimizer 自动切换 paged_adamw_8bit/fused → adamw_torch
- MPS 账本语义：torch.mps 无 max_memory_allocated，peak_alloc 为优化器步采样峰值，
  reserved 用 driver_allocated_memory（与 CUDA reserved 不完全同义，ledger 里标了 device）

用法：
  冒烟：python -m data.train_kev --smoke-steps 30 --data data/dataset/expert_v1.jsonl --out models/kev_smoke
  正式：python -m data.train_kev --epochs 2 --bs 1 --accum 16 --lr 1e-4 \
          --data data/dataset/expert_v1.jsonl --out models/kev_v1
  E1 对照：--no-ckpt / --no-amp / --optim adamw_torch_fused / --quant none
"""
from __future__ import annotations

import json
import math
import os
import random
import time
from collections import Counter


import torch  # noqa: E402
import torch.nn.functional as F  # noqa: E402

LETTERS = "ABCDEFGH"


# ------------------------------------------------------------------ device
def pick_device(name: str) -> str:
    if name != "auto":
        return name
    if torch.cuda.is_available():
        return "cuda"
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return "mps"
    return "cpu"


def mem_alloc_gb(device: str) -> float:
    """CUDA: 真峰值（max_memory_allocated）；MPS: 当前值，由调用方采样取 max。"""
    if device == "cuda":
        return torch.cuda.max_memory_allocated() / 1e9
    if device == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "current_allocated_memory"):
        return torch.mps.current_allocated_memory() / 1e9
    return 0.0


def mem_reserved_gb(device: str) -> float:
    if device == "cuda":
        return torch.cuda.max_memory_reserved() / 1e9
    if device == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "driver_allocated_memory"):
        return torch.mps.driver_allocated_memory() / 1e9
    return 0.0


def reset_mem_peak(device: str) -> None:
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()


def amp_ctx(device: str, enabled: bool):
    """只有 CUDA 需要 autocast；MPS/CPU 上权重本身就是 bf16，直接 nullcontext。"""
    from contextlib import nullcontext
    if enabled and device == "cuda":
        return torch.autocast("cuda", torch.bfloat16)
    return nullcontext()


# ---------------------------------------------------------------------- data
def load_dataset(path: str, seed: int):
    """按对局 seed 切分 90/5/5，防止同一局的状态泄漏到不同集合。"""
    # build_prompt 已在第 5 节定义（notebook 单命名空间）

    by_seed: dict = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            s = json.loads(line)
            by_seed.setdefault(s["seed"], []).append(s)
    seeds = sorted(by_seed)
    random.Random(seed).shuffle(seeds)
    n = len(seeds)
    splits = {"train": seeds[: int(n * 0.9)],
              "dev": seeds[int(n * 0.9): int(n * 0.95)],
              "test": seeds[int(n * 0.95):]}
    out = {}
    for name, ss in splits.items():
        feats = [{"prompt": build_prompt(s["state"]), "label": s["action"]}
                 for k in ss for s in by_seed[k]]
        out[name] = feats
    return out


# --------------------------------------------------------------------- model
def build_model(model_id: str, args, device: str):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model

    quant = None
    if device == "cuda" and args.quant == "nf4":
        from transformers import BitsAndBytesConfig
        quant = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant,
        device_map="cuda" if device == "cuda" else None,
        torch_dtype=torch.bfloat16,
        attn_implementation="sdpa",
        trust_remote_code=True,
    )
    if device != "cuda":
        model.to(device)
    model.config.use_cache = False
    if not args.no_ckpt:
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()

    lora = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()

    # 选项字母 token id（与推理侧取法一致，规范用带空格变体 " A"）
    letter_ids = []
    for ch in LETTERS:
        ids = tok(" " + ch, add_special_tokens=False)["input_ids"]
        assert len(ids) == 1, f'" {ch}" 不是单 token: {ids}'
        letter_ids.append(ids[0])
    return model, tok, letter_ids


def last_position_logits(model, letter_ids, input_ids, attention_mask, use_autocast: bool,
                         device: str):
    """decoder 前向 → 取各样本答案位 hidden → lm_head → 8 字母位 logits。
    绕过 (B,T,V) 全序列 logits，只算需要的那一行（省显存大头）。"""
    with amp_ctx(device, use_autocast):
        try:
            dec = model.get_decoder()
            hs = dec(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        except (AttributeError, TypeError):
            hs = model(input_ids=input_ids, attention_mask=attention_mask,
                       output_hidden_states=True).hidden_states[-1]
        idx = attention_mask.sum(1) - 1  # right padding：最后一个非 pad 位置
        last_h = hs[torch.arange(hs.size(0), device=hs.device), idx]
        logits = model.get_output_embeddings()(last_h)  # (B, V)
    return logits[:, letter_ids]  # (B, 8)


# --------------------------------------------------------------------- train

In [ ]:
@torch.no_grad()
def quick_dev_loss(model, tok, letter_ids, feats, use_autocast: bool, device: str) -> float:
    if not feats:
        return float("nan")
    model.eval()
    enc = tok([f["prompt"] for f in feats], return_tensors="pt",
              padding=True, truncation=True, max_length=768).to(device)
    labels = torch.tensor([f["label"] for f in feats], device=device)
    logits = last_position_logits(model, letter_ids,
                                  enc["input_ids"], enc["attention_mask"], use_autocast,
                                  device=device)
    loss = F.cross_entropy(logits, labels).item()
    model.train()
    return loss

In [ ]:
def train_kev(**overrides):
    """train_kev.py main() 的函数化版本。参数见 defaults，可被 overrides 覆盖。
    例：train_kev(smoke_steps=30)  # 30 步冒烟"""
    from types import SimpleNamespace

    defaults = dict(
        model_id=str(PATHS["base_bf16"]) if PATHS.get("base_bf16") else "Qwen/Qwen3.5-4B",
        data=str(PATHS["dataset"]) if PATHS.get("dataset") else str(RESULTS / "dataset" / "smoke.jsonl"),
        out=str(RESULTS / "models" / "kev_v1"),
        epochs=1.5, bs=4, accum=4, lr=1e-4, warmup=20, log_every=10,
        seed=RUNTIME["seed"], smoke_steps=0, save_every=500,
        no_ckpt=False, no_amp=False, device=RUNTIME["device"],
        quant="nf4", optim="paged_adamw_8bit",
    )
    defaults.update(overrides)
    args = SimpleNamespace(**defaults)

    device = pick_device(args.device)
    if device != "cuda" and args.optim in ("paged_adamw_8bit", "adamw_torch_fused"):
        print(f"[optim] {args.optim} 仅 CUDA 可用，{device} 上自动切换为 adamw_torch")
        args.optim = "adamw_torch"
    if device == "cuda" and (args.quant == "nf4" or args.optim == "paged_adamw_8bit"):
        try:
            import bitsandbytes  # noqa: F401
        except Exception as e:
            print(f"[bnb] bitsandbytes 不可用（{type(e).__name__}: {e}）")
            print("[bnb] 自动降级：quant=none + optim=adamw_torch_fused（bf16 LoRA，24GB 卡装得下）")
            args.quant = "none"
            if args.optim == "paged_adamw_8bit":
                args.optim = "adamw_torch_fused"

    torch.manual_seed(args.seed)
    random.seed(args.seed)

    print(f"[data] loading {args.data} ...")
    ds = load_dataset(args.data, args.seed)
    print(f"[data] train={len(ds['train'])} dev={len(ds['dev'])} test={len(ds['test'])}")
    dist = Counter(f["label"] for f in ds["train"])
    print(f"[data] train 标签分布: {sorted(dist.items())}")

    prec = "NF4 4bit" if (device == "cuda" and args.quant == "nf4") else "bf16"
    print(f"[model] loading {args.model_id} (device={device}, {prec} + LoRA r16) ...")
    model, tok, letter_ids = build_model(args.model_id, args, device)
    print(f"[model] letter_ids={letter_ids}")

    if args.optim == "paged_adamw_8bit":
        import bitsandbytes as bnb
        optim = bnb.optim.PagedAdamW8bit(model.parameters(), lr=args.lr)
    elif args.optim == "adamw_torch_fused":
        optim = torch.optim.AdamW(model.parameters(), lr=args.lr, fused=True)
    else:
        optim = torch.optim.AdamW(model.parameters(), lr=args.lr)

    train = ds["train"]
    random.shuffle(train)
    steps_per_epoch = math.ceil(len(train) / (args.bs * args.accum))
    total_steps = int(steps_per_epoch * args.epochs)
    if args.smoke_steps > 0:
        total_steps = min(total_steps, args.smoke_steps)
    print(f"[train] total optimizer steps = {total_steps} (bs={args.bs} accum={args.accum})")

    def lr_at(step):
        if step < args.warmup:
            return args.lr * (step + 1) / args.warmup
        p = (step - args.warmup) / max(1, total_steps - args.warmup)
        return args.lr * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))

    os.makedirs(args.out, exist_ok=True)
    reset_mem_peak(device)
    ledger = {"config": vars(args), "device": device,
              "mem_semantics": ("cuda: peak=max_memory_allocated, reserved=max_memory_reserved"
                                if device == "cuda" else
                                f"{device}: peak=每步采样 alloc 的 max, reserved=driver_allocated"),
              "steps": []}
    model.train()
    step = micro = 0
    t0 = time.time()
    loss_win, peak_alloc = [], 0.0
    cursor = 0  # 持久游标采样：跨 epoch 边界自然重洗（旧切片 bug 的修复，详见 HANDOFF）

    while step < total_steps:
        batch = []
        while len(batch) < args.bs:
            if cursor >= len(train):
                random.shuffle(train)
                cursor = 0
            take = min(args.bs - len(batch), len(train) - cursor)
            batch.extend(train[cursor:cursor + take])
            cursor += take
        enc = tok([f["prompt"] for f in batch], return_tensors="pt",
                  padding=True, truncation=True, max_length=768).to(device)
        labels = torch.tensor([f["label"] for f in batch], device=device)

        for g in optim.param_groups:
            g["lr"] = lr_at(step)
        logits = last_position_logits(model, letter_ids,
                                      enc["input_ids"], enc["attention_mask"],
                                      use_autocast=not args.no_amp, device=device)
        loss = F.cross_entropy(logits, labels) / args.accum
        loss.backward()
        loss_win.append(loss.item() * args.accum)

        micro += 1
        if micro >= args.accum:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            optim.zero_grad(set_to_none=True)
            micro = 0
            step += 1
            peak_alloc = max(peak_alloc, mem_alloc_gb(device))

            if args.save_every > 0 and step % args.save_every == 0 and step < total_steps:
                snap = os.path.join(args.out, f"ckpt-step{step}")
                model.save_pretrained(snap)
                print(f"  [snapshot] adapter -> {snap}", flush=True)

            if step % args.log_every == 0 or step == total_steps:
                dev_loss = quick_dev_loss(model, tok, letter_ids, ds["dev"][:32],
                                          use_autocast=not args.no_amp, device=device)
                rec = {"step": step,
                       "train_loss": round(sum(loss_win) / len(loss_win), 4),
                       "dev_loss32": round(dev_loss, 4),
                       "lr": lr_at(step),
                       "peak_alloc_gb": round(mem_alloc_gb(device), 2),
                       "peak_reserved_gb": round(mem_reserved_gb(device), 2),
                       "sec_per_step": round((time.time() - t0) / step, 2)}
                ledger["steps"].append(rec)
                loss_win = []
                print(f"  step {step:>4}/{total_steps}  train {rec['train_loss']:.4f}  "
                      f"dev32 {rec['dev_loss32']:.4f}  alloc {rec['peak_alloc_gb']}G  "
                      f"resv {rec['peak_reserved_gb']}G  {rec['sec_per_step']}s/step",
                      flush=True)

    model.save_pretrained(args.out)
    tok.save_pretrained(args.out)
    with open(os.path.join(args.out, "letters.json"), "w") as f:
        json.dump({"letters": LETTERS, "letter_ids": letter_ids}, f)
    ledger["summary"] = {
        "final_train_loss": ledger["steps"][-1]["train_loss"] if ledger["steps"] else None,
        "peak_alloc_gb": round(peak_alloc, 2),
        "peak_reserved_gb": round(mem_reserved_gb(device), 2),
        "total_sec": round(time.time() - t0, 1),
        "total_steps": total_steps,
    }
    with open(os.path.join(args.out, "ledger.json"), "w") as f:
        json.dump(ledger, f, ensure_ascii=False, indent=2)
    print(f"[done] adapter -> {args.out}")
    print(f"[done] E1 账本: peak_alloc={ledger['summary']['peak_alloc_gb']}GB  "
          f"peak_reserved={ledger['summary']['peak_reserved_gb']}GB  "
          f"total={ledger['summary']['total_sec']}s")
    return ledger

### 8.1 运行训练（默认注释掉）

```python
# 30 步冒烟（验证流程，几分钟）：
# train_kev(smoke_steps=30, out=str(RESULTS / "models" / "kev_smoke"))

# 正式训练（36k 数据需配置 paths.dataset；24GB 卡 ~6.6h）：
# train_kev(epochs=1.5, bs=4, accum=4, save_every=500)
```

## 9. 附录：E3 vLLM prefix cache 压测（可选，需 NVIDIA 显卡）

混合架构（Gated DeltaNet）下 vLLM 的 prefix cache 以 **528 tok 页**为复用单位：
共享前缀不满页 = 零命中还倒贴容量；**生效前提 = 页对齐 × 并发**（实测四点曲线 0.54x → 0.97x → 0.98x → **1.61x**）。
完整脚本见原版仓库 `experiments/e3_prefix_cache.py`（支持 `--mode batch --pad-prefix N`），此处不内联——它需要独占整卡，建议在租卡环境跑。

更多实验账本（E1 八组消融 / E2 9B 阶梯 / E4 量化对比）与全部坑记录，见原版仓库 `docs/` 与 `HANDOFF.md`。

---

**收工**。产物都在 `results_root` 里：录像（jsonl + mp4 + 帧）、训练 adapter 与账本、采集的数据集。想深入：原版仓库有完整实验矩阵与技术分享材料。